In [7]:
import pandas as pd
import numpy as np

In [8]:
# ============================================================
# M1: LOAD DATA
# ============================================================

print("Loading featured dataset...")

df = pd.read_csv('../dataset/nft_transfers_jun_aug_ready.csv')

# Fix timestamp
df['timestamp_dt'] = pd.to_datetime(df['timestamp_dt'])

print(f"Shape: {df.shape}")
print(f"Wash trading: {df['is_wash_trading'].sum():,} "
      f"({df['is_wash_trading'].mean():.3%})")

# ============================================================
# CHECK: Distribution by NFT collection
# ============================================================
print("\n=== TOP 20 NFT COLLECTIONS BY TRANSACTION COUNT ===")
collection_stats = df.groupby('nft_address').agg(
    total_tx        = ('transaction_hash', 'count'),
    wash_trading_tx = ('is_wash_trading', 'sum'),
    wash_ratio      = ('is_wash_trading', 'mean')
).sort_values('total_tx', ascending=False).head(20)

print(collection_stats)

print(f"\nTotal unique collections: {df['nft_address'].nunique():,}")

# Check: how many transactions in top N collections?
for n in [10, 20, 50, 100, 200]:
    top_n = df.groupby('nft_address').size(
    ).sort_values(ascending=False).head(n)
    top_n_tx = df[df['nft_address'].isin(top_n.index)]
    wt_in_top = top_n_tx['is_wash_trading'].sum()
    print(f"Top {n:3d} collections: {len(top_n_tx):>8,} tx, "
          f"{wt_in_top:>6,} wash trading "
          f"({top_n_tx['is_wash_trading'].mean():.3%})")

Loading featured dataset...


C:\Users\aryak\AppData\Local\Temp\ipykernel_23064\941515580.py:7: DtypeWarning: Columns (0: token_id) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../dataset/nft_transfers_jun_aug_ready.csv')


Shape: (1563989, 21)
Wash trading: 12,420 (0.794%)

=== TOP 20 NFT COLLECTIONS BY TRANSACTION COUNT ===
                                            total_tx  wash_trading_tx  \
nft_address                                                             
0x57f1887a8BF19b14fC0dF6Fd9B2acc9Af147eA85     88644               11   
0xa7d8d9ef8D8Ce8992Df33D8b8CF4Aebabd5bD270     69001               98   
0xBd3531dA5CF5857e7CfAA92426877b022e612cf8     22647               30   
0x1A92f7381B9F03921564a437210bB9396471050C     18246               41   
0x85f740958906b317de6ed79663012859067E745B     17449               25   
0x06012c8cf97BEaD5deAe237070F9587f8E7A266d     14503               15   
0xbe6e3669464E7dB1e1528212F0BfF5039461CB82     13995                4   
0xe785E82358879F061BC3dcAC6f0444462D4b5330     13577               42   
0x18Df6C571F6fE9283B87f910E41dc5c8b77b7da5     13467               48   
0x3bf2922f4520a8BA0c2eFC3D2a1539678DaD5e9D     12492               30   
0x3F4a885ED8d9cDF10f

In [9]:
# ============================================================
# M1.2: CREATE SUBSET — Top 100 NFT Collections
# Rationale: Most wash trading concentrated in
# high-activity collections (Liu et al., 2023)
# ============================================================

print("Creating subset: Top 100 NFT collections...")

# Get top 100 collections by transaction count
top_100_collections = df.groupby(
    'nft_address'
).size().sort_values(ascending=False).head(100).index

# Filter dataset
df_subset = df[
    df['nft_address'].isin(top_100_collections)
].copy().reset_index(drop=True)

print(f"Subset shape      : {df_subset.shape}")
print(f"Total tx          : {len(df_subset):,}")
print(f"Wash trading      : {df_subset['is_wash_trading'].sum():,} "
      f"({df_subset['is_wash_trading'].mean():.3%})")
print(f"Unique wallets    : "
      f"{pd.concat([df_subset['from_address'], df_subset['to_address']]).nunique():,}")
print(f"Unique collections: {df_subset['nft_address'].nunique():,}")

Creating subset: Top 100 NFT collections...
Subset shape      : (986923, 21)
Total tx          : 986,923
Wash trading      : 7,827 (0.793%)
Unique wallets    : 152,016
Unique collections: 100


In [10]:
# ============================================================
# M1.3: TEMPORAL SPLIT 70/30
# Train: earlier 70% by timestamp
# Test : later 30% by timestamp
# ============================================================
print("\n=== TEMPORAL SPLIT 70/30 ===")

df_subset = df_subset.sort_values(
    'timestamp'
).reset_index(drop=True)

split_idx  = int(len(df_subset) * 0.70)
split_time = df_subset.loc[split_idx, 'timestamp_dt']

train = df_subset.iloc[:split_idx].copy()
test  = df_subset.iloc[split_idx:].copy()

print(f"Split point : {split_time}")
print(f"\nTrain: {len(train):,} rows ({len(train)/len(df_subset):.1%})")
print(f"  Period: {train['timestamp_dt'].min()} "
      f"to {train['timestamp_dt'].max()}")
print(f"  Wash trading: {train['is_wash_trading'].sum():,} "
      f"({train['is_wash_trading'].mean():.3%})")
print(f"\nTest : {len(test):,} rows ({len(test)/len(df_subset):.1%})")
print(f"  Period: {test['timestamp_dt'].min()} "
      f"to {test['timestamp_dt'].max()}")
print(f"  Wash trading: {test['is_wash_trading'].sum():,} "
      f"({test['is_wash_trading'].mean():.3%})")


=== TEMPORAL SPLIT 70/30 ===
Split point : 2021-08-20 04:59:34

Train: 690,846 rows (70.0%)
  Period: 2021-06-01 00:00:26 to 2021-08-20 04:59:06
  Wash trading: 6,206 (0.898%)

Test : 296,077 rows (30.0%)
  Period: 2021-08-20 04:59:34 to 2021-08-31 23:59:55
  Wash trading: 1,621 (0.547%)


In [11]:
# ============================================================
# M1.4: DEFINE FEATURES
# ============================================================
FEATURE_COLS = [
    'transaction_value',
    'holding_time_hours',
    'pair_frequency',
    'time_since_last_hours',
    'symmetry_ratio_from',
    'transfers_out_from',
    'transfers_in_from',
    'transfers_out_to',
    'transfers_in_to',
    'num_transitions',
]

LABEL_COL = 'is_wash_trading'

# Prepare X, y
X_train = train[FEATURE_COLS].values
y_train = train[LABEL_COL].values
X_test  = test[FEATURE_COLS].values
y_test  = test[LABEL_COL].values

print(f"\n=== FEATURE MATRIX ===")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape : {X_test.shape}")
print(f"y_train — wash: {y_train.sum():,} "
      f"({y_train.mean():.3%})")
print(f"y_test  — wash: {y_test.sum():,} "
      f"({y_test.mean():.3%})")

# Class weight for imbalanced data
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=y_train
)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}

print(f"\n=== CLASS WEIGHTS ===")
print(f"Normal (0)       : {class_weights[0]:.4f}")
print(f"Wash trading (1) : {class_weights[1]:.4f}")
print(f"Ratio            : 1 : {class_weights[1]/class_weights[0]:.0f}")


=== FEATURE MATRIX ===
X_train shape: (690846, 10)
X_test shape : (296077, 10)
y_train — wash: 6,206 (0.898%)
y_test  — wash: 1,621 (0.547%)

=== CLASS WEIGHTS ===
Normal (0)       : 0.5045
Wash trading (1) : 55.6595
Ratio            : 1 : 110


In [12]:
# ============================================================
# M2: RANDOM FOREST (Baseline Tabular)
# No graph structure, no temporal modeling
# Question: "What can standard ML do without graph?"
# ============================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score,
    classification_report, confusion_matrix
)
import time

print("=" * 55)
print("M2: RANDOM FOREST (Baseline Tabular)")
print("=" * 55)

# ============================================================
# M2.1: TRAIN
# ============================================================
print("\nTraining Random Forest...")
print(f"  n_estimators : 100")
print(f"  class_weight : balanced")
print(f"  n_jobs       : -1 (all CPU cores)")

start_time = time.time()

rf_model = RandomForestClassifier(
    n_estimators  = 100,
    class_weight  = 'balanced',  # handle imbalance
    random_state  = 42,
    n_jobs        = -1           # use all CPU cores
)

rf_model.fit(X_train, y_train)
train_time = time.time() - start_time

print(f"  Training time: {train_time:.2f} seconds")

# ============================================================
# M2.2: PREDICT
# ============================================================
print("\nPredicting...")

y_pred_rf       = rf_model.predict(X_test)
y_pred_proba_rf = rf_model.predict_proba(X_test)[:, 1]

# ============================================================
# M2.3: EVALUATE
# ============================================================
print("\n=== RANDOM FOREST RESULTS ===")

f1_rf        = f1_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf)
recall_rf    = recall_score(y_test, y_pred_rf)
roc_auc_rf   = roc_auc_score(y_test, y_pred_proba_rf)
pr_auc_rf    = average_precision_score(y_test, y_pred_proba_rf)

print(f"  F1 Score   : {f1_rf:.4f}")
print(f"  Precision  : {precision_rf:.4f}")
print(f"  Recall     : {recall_rf:.4f}")
print(f"  ROC-AUC    : {roc_auc_rf:.4f}")
print(f"  PR-AUC     : {pr_auc_rf:.4f}")
print(f"  Train time : {train_time:.2f}s")

# Confusion matrix
cm_rf = confusion_matrix(y_test, y_pred_rf)
print(f"\nConfusion Matrix:")
print(f"                 Predicted Normal  Predicted Wash")
print(f"Actual Normal  : {cm_rf[0][0]:>15,}  {cm_rf[0][1]:>14,}")
print(f"Actual Wash    : {cm_rf[1][0]:>15,}  {cm_rf[1][1]:>14,}")

tn, fp, fn, tp = cm_rf.ravel()
print(f"\n  True Negative  (correct normal)  : {tn:,}")
print(f"  False Positive (normal → wash)   : {fp:,}")
print(f"  False Negative (wash → normal)   : {fn:,}")
print(f"  True Positive  (correct wash)    : {tp:,}")

# Classification report
print(f"\nClassification Report:")
print(classification_report(
    y_test, y_pred_rf,
    target_names=['Normal', 'Wash Trading']
))

# ============================================================
# M2.4: FEATURE IMPORTANCE
# ============================================================
print("\n=== FEATURE IMPORTANCE ===")
importance_df = pd.DataFrame({
    'feature'   : FEATURE_COLS,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

for _, row in importance_df.iterrows():
    bar = '█' * int(row['importance'] * 50)
    print(f"  {row['feature']:<25} {row['importance']:.4f} {bar}")

# Store results for final comparison
rf_results = {
    'model'    : 'Random Forest',
    'f1'       : f1_rf,
    'precision': precision_rf,
    'recall'   : recall_rf,
    'roc_auc'  : roc_auc_rf,
    'pr_auc'   : pr_auc_rf,
    'train_time': train_time
}

print("\nRandom Forest done ✅")
print("Next: M3 — GCN")

M2: RANDOM FOREST (Baseline Tabular)

Training Random Forest...
  n_estimators : 100
  class_weight : balanced
  n_jobs       : -1 (all CPU cores)
  Training time: 49.36 seconds

Predicting...

=== RANDOM FOREST RESULTS ===
  F1 Score   : 0.8325
  Precision  : 0.9799
  Recall     : 0.7236
  ROC-AUC    : 0.9580
  PR-AUC     : 0.8820
  Train time : 49.36s

Confusion Matrix:
                 Predicted Normal  Predicted Wash
Actual Normal  :         294,432              24
Actual Wash    :             448           1,173

  True Negative  (correct normal)  : 294,432
  False Positive (normal → wash)   : 24
  False Negative (wash → normal)   : 448
  True Positive  (correct wash)    : 1,173

Classification Report:
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00    294456
Wash Trading       0.98      0.72      0.83      1621

    accuracy                           1.00    296077
   macro avg       0.99      0.86      0.92    296077
weighted av

In [13]:
# ============================================================
# M2.2: LOGISTIC REGRESSION (Linear Baseline)
# Simplest possible baseline — linear decision boundary
# Reference: Standard practice in fraud detection literature
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

print("=" * 55)
print("M2.2: LOGISTIC REGRESSION (Linear Baseline)")
print("=" * 55)

# Logistic Regression needs feature scaling
print("\nScaling features...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("\nTraining Logistic Regression...")
start_time = time.time()

lr_model = LogisticRegression(
    class_weight = 'balanced',
    max_iter     = 1000,
    random_state = 42,
    n_jobs       = -1
)

lr_model.fit(X_train_scaled, y_train)
train_time_lr = time.time() - start_time
print(f"  Training time: {train_time_lr:.2f} seconds")

# Predict
y_pred_lr       = lr_model.predict(X_test_scaled)
y_pred_proba_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

# Evaluate
print("\n=== LOGISTIC REGRESSION RESULTS ===")
f1_lr        = f1_score(y_test, y_pred_lr)
precision_lr = precision_score(y_test, y_pred_lr)
recall_lr    = recall_score(y_test, y_pred_lr)
roc_auc_lr   = roc_auc_score(y_test, y_pred_proba_lr)
pr_auc_lr    = average_precision_score(y_test, y_pred_proba_lr)

print(f"  F1 Score   : {f1_lr:.4f}")
print(f"  Precision  : {precision_lr:.4f}")
print(f"  Recall     : {recall_lr:.4f}")
print(f"  ROC-AUC    : {roc_auc_lr:.4f}")
print(f"  PR-AUC     : {pr_auc_lr:.4f}")
print(f"  Train time : {train_time_lr:.2f}s")

cm_lr = confusion_matrix(y_test, y_pred_lr)
tn, fp, fn, tp = cm_lr.ravel()
print(f"\nConfusion Matrix:")
print(f"                 Predicted Normal  Predicted Wash")
print(f"Actual Normal  : {cm_lr[0][0]:>15,}  {cm_lr[0][1]:>14,}")
print(f"Actual Wash    : {cm_lr[1][0]:>15,}  {cm_lr[1][1]:>14,}")
print(f"\n  TN: {tn:,} | FP: {fp:,} | FN: {fn:,} | TP: {tp:,}")

lr_results = {
    'model'     : 'Logistic Regression',
    'f1'        : f1_lr,
    'precision' : precision_lr,
    'recall'    : recall_lr,
    'roc_auc'   : roc_auc_lr,
    'pr_auc'    : pr_auc_lr,
    'train_time': train_time_lr
}

print("\nLogistic Regression done ✅")

# ============================================================
# M2.3: XGBOOST (Strong Tabular Baseline)
# Gradient boosting — state-of-the-art for tabular data
# Reference: Falk et al. (2023) use gradient boosting
# for NFT fraud detection
# ============================================================

print("\n" + "=" * 55)
print("M2.3: XGBOOST (Strong Tabular Baseline)")
print("=" * 55)

try:
    from xgboost import XGBClassifier
    xgb_available = True
except ImportError:
    print("XGBoost not installed. Installing...")
    import subprocess
    subprocess.run(['pip', 'install', 'xgboost'], 
                  capture_output=True)
    from xgboost import XGBClassifier
    xgb_available = True

# Calculate scale_pos_weight for XGBoost
# = number of negative samples / number of positive samples
n_negative = (y_train == 0).sum()
n_positive = (y_train == 1).sum()
scale_pos_weight = n_negative / n_positive

print(f"\nXGBoost class imbalance handling:")
print(f"  scale_pos_weight: {scale_pos_weight:.2f}")
print(f"  (negative/positive = {n_negative:,}/{n_positive:,})")

print("\nTraining XGBoost...")
start_time = time.time()

xgb_model = XGBClassifier(
    n_estimators      = 100,
    scale_pos_weight  = scale_pos_weight,
    random_state      = 42,
    n_jobs            = -1,
    eval_metric       = 'logloss',
    verbosity         = 0
)

xgb_model.fit(X_train, y_train)
train_time_xgb = time.time() - start_time
print(f"  Training time: {train_time_xgb:.2f} seconds")

# Predict
y_pred_xgb       = xgb_model.predict(X_test)
y_pred_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

# Evaluate
print("\n=== XGBOOST RESULTS ===")
f1_xgb        = f1_score(y_test, y_pred_xgb)
precision_xgb = precision_score(y_test, y_pred_xgb)
recall_xgb    = recall_score(y_test, y_pred_xgb)
roc_auc_xgb   = roc_auc_score(y_test, y_pred_proba_xgb)
pr_auc_xgb    = average_precision_score(y_test, y_pred_proba_xgb)

print(f"  F1 Score   : {f1_xgb:.4f}")
print(f"  Precision  : {precision_xgb:.4f}")
print(f"  Recall     : {recall_xgb:.4f}")
print(f"  ROC-AUC    : {roc_auc_xgb:.4f}")
print(f"  PR-AUC     : {pr_auc_xgb:.4f}")
print(f"  Train time : {train_time_xgb:.2f}s")

cm_xgb = confusion_matrix(y_test, y_pred_xgb)
tn, fp, fn, tp = cm_xgb.ravel()
print(f"\nConfusion Matrix:")
print(f"                 Predicted Normal  Predicted Wash")
print(f"Actual Normal  : {cm_xgb[0][0]:>15,}  {cm_xgb[0][1]:>14,}")
print(f"Actual Wash    : {cm_xgb[1][0]:>15,}  {cm_xgb[1][1]:>14,}")
print(f"\n  TN: {tn:,} | FP: {fp:,} | FN: {fn:,} | TP: {tp:,}")

# Feature importance
print("\n=== XGBOOST FEATURE IMPORTANCE ===")
xgb_importance = pd.DataFrame({
    'feature'   : FEATURE_COLS,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

for _, row in xgb_importance.iterrows():
    bar = '█' * int(row['importance'] * 50)
    print(f"  {row['feature']:<25} {row['importance']:.4f} {bar}")

xgb_results = {
    'model'     : 'XGBoost',
    'f1'        : f1_xgb,
    'precision' : precision_xgb,
    'recall'    : recall_xgb,
    'roc_auc'   : roc_auc_xgb,
    'pr_auc'    : pr_auc_xgb,
    'train_time': train_time_xgb
}

print("\nXGBoost done ✅")

# ============================================================
# TABULAR BASELINES SUMMARY
# ============================================================
print("\n" + "=" * 55)
print("TABULAR BASELINES SUMMARY")
print("=" * 55)

results_so_far = [lr_results, rf_results, xgb_results]
summary_df = pd.DataFrame(results_so_far).set_index('model')

print(summary_df[[
    'f1', 'precision', 'recall',
    'roc_auc', 'pr_auc', 'train_time'
]].to_string())

print("\nAll tabular baselines done ✅")
print("Next: M3 — GCN")

M2.2: LOGISTIC REGRESSION (Linear Baseline)

Scaling features...

Training Logistic Regression...


c:\Users\aryak\anaconda3\envs\nft-research\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


  Training time: 2.66 seconds

=== LOGISTIC REGRESSION RESULTS ===
  F1 Score   : 0.2934
  Precision  : 0.1765
  Recall     : 0.8698
  ROC-AUC    : 0.9624
  PR-AUC     : 0.8087
  Train time : 2.66s

Confusion Matrix:
                 Predicted Normal  Predicted Wash
Actual Normal  :         287,876           6,580
Actual Wash    :             211           1,410

  TN: 287,876 | FP: 6,580 | FN: 211 | TP: 1,410

Logistic Regression done ✅

M2.3: XGBOOST (Strong Tabular Baseline)

XGBoost class imbalance handling:
  scale_pos_weight: 110.32
  (negative/positive = 684,640/6,206)

Training XGBoost...
  Training time: 2.80 seconds

=== XGBOOST RESULTS ===
  F1 Score   : 0.8105
  Precision  : 0.7792
  Recall     : 0.8445
  ROC-AUC    : 0.9723
  PR-AUC     : 0.8550
  Train time : 2.80s

Confusion Matrix:
                 Predicted Normal  Predicted Wash
Actual Normal  :         294,068             388
Actual Wash    :             252           1,369

  TN: 294,068 | FP: 388 | FN: 252 | TP: 1,

In [15]:
# ============================================================
# M3: GCN FIXED — Proper Temporal Graph Split
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score,
    confusion_matrix
)
import time
import numpy as np

print("=" * 55)
print("M3: GCN — Proper Temporal Graph Split")
print("=" * 55)

# ============================================================
# KEY FIX: Build SEPARATE graphs for train and test
# Train graph: only edges from training period
# Test graph : train edges + test edges (inductive)
# Node features computed from training data only
# ============================================================

# Split dataframe first
split_idx  = int(len(df_subset) * 0.70)
train_df   = df_subset.iloc[:split_idx].copy()
test_df    = df_subset.iloc[split_idx:].copy()

print(f"Train edges: {len(train_df):,}")
print(f"Test edges : {len(test_df):,}")

# ============================================================
# BUILD WALLET INDEX from TRAIN only
# (avoid leaking test wallet info)
# ============================================================
train_wallets   = pd.concat([
    train_df['from_address'],
    train_df['to_address']
]).unique()
wallet_to_idx   = {w: i for i, w in enumerate(train_wallets)}
n_nodes_train   = len(wallet_to_idx)
print(f"\nWallets in train: {n_nodes_train:,}")

# For test: some wallets might be new (unseen in train)
# We handle this by adding them to the index
test_wallets_new = set(
    pd.concat([
        test_df['from_address'],
        test_df['to_address']
    ]).unique()
) - set(wallet_to_idx.keys())

print(f"New wallets in test: {len(test_wallets_new):,}")

# Add new wallets to index
for w in test_wallets_new:
    wallet_to_idx[w] = len(wallet_to_idx)
n_nodes_total = len(wallet_to_idx)
print(f"Total wallets: {n_nodes_total:,}")

# ============================================================
# NODE FEATURES — computed from TRAIN data only
# ============================================================
print("\nComputing node features from train data only...")

node_feature_cols = [
    'transfers_out_from',
    'transfers_in_from',
    'symmetry_ratio_from',
]

from_feat = train_df.groupby('from_address')[
    node_feature_cols
].mean()
from_feat.columns = [
    'transfers_out', 'transfers_in', 'symmetry_ratio'
]

to_feat = train_df.groupby('to_address')[[
    'transfers_out_to', 'transfers_in_to'
]].mean()
to_feat.columns = ['transfers_out', 'transfers_in']

# Build node feature matrix
all_wallet_list = list(wallet_to_idx.keys())
node_df = pd.DataFrame(
    index=all_wallet_list,
    columns=['transfers_out', 'transfers_in', 'symmetry_ratio'],
    dtype=float
)
node_df.update(from_feat)
missing = node_df['transfers_out'].isna()
node_df.loc[missing, ['transfers_out', 'transfers_in']] = \
    to_feat.reindex(node_df[missing].index).values
node_df = node_df.fillna(0)

# Normalize
node_scaler = StandardScaler()
node_arr    = node_scaler.fit_transform(node_df.values)
x           = torch.tensor(node_arr, dtype=torch.float)
print(f"  Node features: {x.shape}")

# ============================================================
# EDGE FEATURES — normalized from TRAIN data
# ============================================================
edge_feature_cols = [
    'transaction_value',
    'holding_time_hours',
    'time_since_last_hours',
    'pair_frequency',
    'num_transitions',
]

# Fit scaler on TRAIN only → transform both train and test
edge_scaler = StandardScaler()
train_edge_arr = edge_scaler.fit_transform(
    train_df[edge_feature_cols].values.astype(float)
)
test_edge_arr  = edge_scaler.transform(
    test_df[edge_feature_cols].values.astype(float)
)

# ============================================================
# BUILD TRAIN GRAPH
# ============================================================
print("\nBuilding train graph...")

train_src = train_df['from_address'].map(wallet_to_idx).values
train_dst = train_df['to_address'].map(wallet_to_idx).values

train_edge_index = torch.tensor(
    np.array([train_src, train_dst]), dtype=torch.long
)
train_edge_attr  = torch.tensor(
    train_edge_arr, dtype=torch.float
)
train_labels     = torch.tensor(
    train_df['is_wash_trading'].values, dtype=torch.long
)

print(f"  Train graph: {train_edge_index.shape[1]:,} edges")
print(f"  Wash trading: {train_labels.sum().item():,}")

# ============================================================
# BUILD TEST GRAPH
# ============================================================
print("Building test graph...")

test_src = test_df['from_address'].map(wallet_to_idx).values
test_dst = test_df['to_address'].map(wallet_to_idx).values

test_edge_index = torch.tensor(
    np.array([test_src, test_dst]), dtype=torch.long
)
test_edge_attr  = torch.tensor(
    test_edge_arr, dtype=torch.float
)
test_labels     = torch.tensor(
    test_df['is_wash_trading'].values, dtype=torch.long
)

print(f"  Test graph: {test_edge_index.shape[1]:,} edges")
print(f"  Wash trading: {test_labels.sum().item():,}")

# ============================================================
# GCN MODEL (same architecture)
# ============================================================
class GCN(nn.Module):
    def __init__(self,
                 node_feat_dim,
                 edge_feat_dim,
                 hidden_dim=64,
                 dropout=0.3):
        super(GCN, self).__init__()
        self.conv1 = GCNConv(node_feat_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.edge_classifier = nn.Sequential(
            nn.Linear(
                hidden_dim * 2 + edge_feat_dim, hidden_dim
            ),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 2)
        )
        self.dropout = dropout

    def forward(self, x, edge_index, edge_attr):
        h = self.conv1(x, edge_index)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout,
                      training=self.training)
        h = self.conv2(h, edge_index)
        h = F.relu(h)
        src_idx   = edge_index[0]
        dst_idx   = edge_index[1]
        edge_repr = torch.cat([
            h[src_idx], h[dst_idx], edge_attr
        ], dim=1)
        return self.edge_classifier(edge_repr)

# ============================================================
# TRAINING
# ============================================================
device = torch.device('cpu')

# Stable class weight
n_pos        = train_labels.sum().item()
n_neg        = len(train_labels) - n_pos
stable_weight = np.sqrt(n_neg / n_pos)
print(f"\nClass weight (stable): {stable_weight:.2f}")

criterion = nn.CrossEntropyLoss(
    weight=torch.tensor([1.0, stable_weight],
                        dtype=torch.float)
)

model_gcn = GCN(
    node_feat_dim = x.shape[1],
    edge_feat_dim = train_edge_attr.shape[1],
    hidden_dim    = 64,
    dropout       = 0.3
).to(device)

optimizer = torch.optim.Adam(
    model_gcn.parameters(),
    lr=0.001, weight_decay=1e-5
)

# Move to device
x_dev              = x.to(device)
train_edge_idx_dev = train_edge_index.to(device)
train_edge_attr_dev = train_edge_attr.to(device)
train_labels_dev   = train_labels.to(device)
test_edge_idx_dev  = test_edge_index.to(device)
test_edge_attr_dev = test_edge_attr.to(device)

print(f"\nTraining GCN (50 epochs)...")
N_EPOCHS   = 50
start_time = time.time()
best_f1    = 0
best_state = None

for epoch in range(1, N_EPOCHS + 1):

    # TRAIN on train graph
    model_gcn.train()
    optimizer.zero_grad()
    out  = model_gcn(
        x_dev, train_edge_idx_dev, train_edge_attr_dev
    )
    loss = criterion(out, train_labels_dev)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(
        model_gcn.parameters(), max_norm=1.0
    )
    optimizer.step()

    if epoch % 10 == 0:
        # Evaluate on TEST graph
        model_gcn.eval()
        with torch.no_grad():
            # Train metrics
            pred_train = out.argmax(dim=1)
            f1_train   = f1_score(
                train_labels.numpy(),
                pred_train.cpu().numpy(),
                zero_division=0
            )
            # Test metrics
            out_test  = model_gcn(
                x_dev, test_edge_idx_dev, test_edge_attr_dev
            )
            pred_test = out_test.argmax(dim=1)
            f1_test   = f1_score(
                test_labels.numpy(),
                pred_test.cpu().numpy(),
                zero_division=0
            )

            # Save best model
            if f1_test > best_f1:
                best_f1    = f1_test
                best_state = {
                    k: v.clone()
                    for k, v in model_gcn.state_dict().items()
                }

        print(f"  Epoch {epoch:3d}/{N_EPOCHS} "
              f"| Loss: {loss.item():.4f} "
              f"| Train F1: {f1_train:.4f} "
              f"| Test F1: {f1_test:.4f}")

train_time_gcn = time.time() - start_time
print(f"\nTraining time: {train_time_gcn:.2f}s")
print(f"Best Test F1 : {best_f1:.4f}")

# ============================================================
# EVALUATION — use best model
# ============================================================
print("\nEvaluating best GCN model...")

if best_state is not None:
    model_gcn.load_state_dict(best_state)

model_gcn.eval()
with torch.no_grad():
    out_test = model_gcn(
        x_dev, test_edge_idx_dev, test_edge_attr_dev
    )
    y_pred_gcn       = out_test.argmax(dim=1).cpu().numpy()
    y_pred_proba_gcn = torch.softmax(
        out_test, dim=1
    )[:, 1].cpu().numpy()
    y_true_gcn       = test_labels.cpu().numpy()

f1_gcn        = f1_score(y_true_gcn, y_pred_gcn,
                          zero_division=0)
precision_gcn = precision_score(y_true_gcn, y_pred_gcn,
                                 zero_division=0)
recall_gcn    = recall_score(y_true_gcn, y_pred_gcn,
                              zero_division=0)
roc_auc_gcn   = roc_auc_score(y_true_gcn, y_pred_proba_gcn)
pr_auc_gcn    = average_precision_score(
    y_true_gcn, y_pred_proba_gcn
)

print(f"\n=== GCN RESULTS ===")
print(f"  F1 Score   : {f1_gcn:.4f}")
print(f"  Precision  : {precision_gcn:.4f}")
print(f"  Recall     : {recall_gcn:.4f}")
print(f"  ROC-AUC    : {roc_auc_gcn:.4f}")
print(f"  PR-AUC     : {pr_auc_gcn:.4f}")
print(f"  Train time : {train_time_gcn:.2f}s")

cm_gcn = confusion_matrix(y_true_gcn, y_pred_gcn)
tn, fp, fn, tp = cm_gcn.ravel()
print(f"\nConfusion Matrix:")
print(f"                 Predicted Normal  Predicted Wash")
print(f"Actual Normal  : {cm_gcn[0][0]:>15,}  {cm_gcn[0][1]:>14,}")
print(f"Actual Wash    : {cm_gcn[1][0]:>15,}  {cm_gcn[1][1]:>14,}")
print(f"\n  TN: {tn:,} | FP: {fp:,} | FN: {fn:,} | TP: {tp:,}")

gcn_results = {
    'model'     : 'GCN',
    'f1'        : f1_gcn,
    'precision' : precision_gcn,
    'recall'    : recall_gcn,
    'roc_auc'   : roc_auc_gcn,
    'pr_auc'    : pr_auc_gcn,
    'train_time': train_time_gcn
}

# Comparison
print(f"\n{'='*55}")
print(f"RESULTS COMPARISON")
print(f"{'='*55}")
all_results   = [
    lr_results, rf_results, xgb_results, gcn_results
]
comparison_df = pd.DataFrame(
    all_results
).set_index('model')
print(comparison_df[[
    'f1', 'precision', 'recall', 'roc_auc', 'pr_auc'
]].to_string())

print("\nGCN done ✅")
print("Next: M4 — TGN")

M3: GCN — Proper Temporal Graph Split
Train edges: 690,846
Test edges : 296,077

Wallets in train: 107,050
New wallets in test: 44,966
Total wallets: 152,016

Computing node features from train data only...
  Node features: torch.Size([152016, 3])

Building train graph...
  Train graph: 690,846 edges
  Wash trading: 6,206
Building test graph...
  Test graph: 296,077 edges
  Wash trading: 1,621

Class weight (stable): 10.50

Training GCN (50 epochs)...
  Epoch  10/50 | Loss: 0.4212 | Train F1: 0.0195 | Test F1: 0.0000
  Epoch  20/50 | Loss: 0.2496 | Train F1: 0.4753 | Test F1: 0.0000
  Epoch  30/50 | Loss: 0.2197 | Train F1: 0.6719 | Test F1: 0.0011
  Epoch  40/50 | Loss: 0.2088 | Train F1: 0.7013 | Test F1: 0.0106
  Epoch  50/50 | Loss: 0.1835 | Train F1: 0.6602 | Test F1: 0.0106

Training time: 184.11s
Best Test F1 : 0.0106

Evaluating best GCN model...

=== GCN RESULTS ===
  F1 Score   : 0.0106
  Precision  : 0.0389
  Recall     : 0.0062
  ROC-AUC    : 0.5925
  PR-AUC     : 0.0066
  

In [16]:
# Cek: apakah wash trading di test set
# melibatkan wallet baru?

test_wt = test_df[test_df['is_wash_trading'] == 1]
train_wallet_set = set(pd.concat([
    train_df['from_address'],
    train_df['to_address']
]).unique())

# Check from_address
new_from = test_wt[
    ~test_wt['from_address'].isin(train_wallet_set)
]
new_to = test_wt[
    ~test_wt['to_address'].isin(train_wallet_set)
]

print(f"Wash trading in test: {len(test_wt):,}")
print(f"With new from_address: {len(new_from):,} "
      f"({len(new_from)/len(test_wt):.1%})")
print(f"With new to_address  : {len(new_to):,} "
      f"({len(new_to)/len(test_wt):.1%})")
print(f"\nNew wallets in test (total): {len(test_wallets_new):,} "
      f"({len(test_wallets_new)/len(train_wallets):.1%} of train wallets)")

Wash trading in test: 1,621
With new from_address: 101 (6.2%)
With new to_address  : 140 (8.6%)

New wallets in test (total): 44,966 (42.0% of train wallets)


In [17]:
# Verifikasi: apakah masalahnya di graph connectivity?
print("=== GRAPH CONNECTIVITY ANALYSIS ===")

# Train graph stats
train_degree = pd.concat([
    train_df['from_address'],
    train_df['to_address']
]).value_counts()

# Test graph stats
test_degree = pd.concat([
    test_df['from_address'],
    test_df['to_address']
]).value_counts()

# Wash trading wallet degrees in train
wt_wallets_train = set(pd.concat([
    train_df[train_df['is_wash_trading']==1]['from_address'],
    train_df[train_df['is_wash_trading']==1]['to_address']
]).unique())

wt_wallets_test = set(pd.concat([
    test_df[test_df['is_wash_trading']==1]['from_address'],
    test_df[test_df['is_wash_trading']==1]['to_address']
]).unique())

overlap = wt_wallets_train.intersection(wt_wallets_test)

print(f"Wash trading wallets in train: {len(wt_wallets_train):,}")
print(f"Wash trading wallets in test : {len(wt_wallets_test):,}")
print(f"Overlap (same wallets)       : {len(overlap):,} "
      f"({len(overlap)/len(wt_wallets_test):.1%})")
print(f"\nAvg degree in train: {train_degree.mean():.2f}")
print(f"Avg degree in test : {test_degree.mean():.2f}")

=== GRAPH CONNECTIVITY ANALYSIS ===
Wash trading wallets in train: 850
Wash trading wallets in test : 270
Overlap (same wallets)       : 50 (18.5%)

Avg degree in train: 12.91
Avg degree in test : 6.55


In [18]:
# ============================================================
# M4: TGN (Temporal Graph Network)
# Main model — graph + temporal dynamics
# Reference: Rossi et al. (2020) arxiv:2006.10637
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score,
    confusion_matrix
)
import time
import numpy as np

print("=" * 55)
print("M4: TGN (Temporal Graph Network)")
print("=" * 55)

# ============================================================
# M4.1: PREPARE TEMPORAL DATA
# TGN needs: src, dst, timestamps, edge_features, labels
# Sorted by timestamp (critical for TGN!)
# ============================================================
print("\nPreparing temporal data...")

# Use same train/test split as GCN
# train_df and test_df already defined

# Normalize timestamps to [0, 1] for numerical stability
all_timestamps = df_subset['timestamp'].values
ts_min = all_timestamps.min()
ts_max = all_timestamps.max()

def normalize_ts(ts):
    return (ts - ts_min) / (ts_max - ts_min + 1e-8)

train_ts = normalize_ts(train_df['timestamp'].values)
test_ts  = normalize_ts(test_df['timestamp'].values)

# Source and destination node indices
train_src = train_df['from_address'].map(wallet_to_idx).values
train_dst = train_df['to_address'].map(wallet_to_idx).values
test_src  = test_df['from_address'].map(wallet_to_idx).values
test_dst  = test_df['to_address'].map(wallet_to_idx).values

# Edge features (already normalized from GCN section)
# train_edge_arr and test_edge_arr already defined

# Labels
train_labels_np = train_df['is_wash_trading'].values
test_labels_np  = test_df['is_wash_trading'].values

print(f"Train: {len(train_src):,} events")
print(f"Test : {len(test_src):,} events")
print(f"Nodes: {n_nodes_total:,}")
print(f"Edge feature dim: {train_edge_arr.shape[1]}")

# ============================================================
# M4.2: TGN MODEL DEFINITION
# ============================================================

class TGN(nn.Module):
    def __init__(self,
                 n_nodes,
                 edge_feat_dim,
                 memory_dim   = 64,
                 hidden_dim   = 64,
                 dropout      = 0.3):
        super(TGN, self).__init__()

        self.n_nodes    = n_nodes
        self.memory_dim = memory_dim

        # Memory: one vector per node
        # Stores historical behavior of each wallet
        self.memory = nn.Parameter(
            torch.zeros(n_nodes, memory_dim),
            requires_grad=False
        )
        self.last_update = torch.zeros(n_nodes)

        # Message function:
        # Compute message from (src_memory, dst_memory,
        #                       edge_features, time_delta)
        msg_dim = memory_dim * 2 + edge_feat_dim + 1
        self.msg_fn = nn.Sequential(
            nn.Linear(msg_dim, memory_dim),
            nn.ReLU()
        )

        # Memory updater (GRU)
        # Updates node memory based on incoming messages
        self.memory_updater = nn.GRUCell(
            input_size  = memory_dim,
            hidden_size = memory_dim
        )

        # Edge classifier
        # Input: src_memory + dst_memory + edge_features
        classifier_input = memory_dim * 2 + edge_feat_dim
        self.edge_classifier = nn.Sequential(
            nn.Linear(classifier_input, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 2)
        )

    def reset_memory(self):
        """Reset memory at start of each epoch"""
        nn.init.zeros_(self.memory)
        self.last_update = torch.zeros(self.n_nodes)

    def forward(self,
                src_nodes,
                dst_nodes,
                timestamps,
                edge_features,
                update_memory=True):
        """
        Process a batch of temporal edges.

        Args:
            src_nodes    : source wallet indices [B]
            dst_nodes    : destination wallet indices [B]
            timestamps   : normalized timestamps [B]
            edge_features: edge feature matrix [B, F]
            update_memory: whether to update memory
                          (True during training,
                           False during inference)
        """

        # Get current memory of source and destination
        src_mem = self.memory[src_nodes]  # [B, memory_dim]
        dst_mem = self.memory[dst_nodes]  # [B, memory_dim]

        # Time delta since last update
        src_time_delta = (
            timestamps - self.last_update[src_nodes]
        ).unsqueeze(1)                    # [B, 1]

        # Compute messages
        # Message contains: what happened in this interaction
        msg_input = torch.cat([
            src_mem,
            dst_mem,
            edge_features,
            src_time_delta
        ], dim=1)                         # [B, msg_dim]

        msg = self.msg_fn(msg_input)      # [B, memory_dim]

        # Update memory if training
        if update_memory:
            # Update source node memory
            new_src_mem = self.memory_updater(
                msg, src_mem
            )
            # Update destination node memory
            new_dst_mem = self.memory_updater(
                msg, dst_mem
            )

            # Write back to memory
            self.memory.data[src_nodes] = new_src_mem.detach()
            self.memory.data[dst_nodes] = new_dst_mem.detach()

            # Update last interaction timestamp
            self.last_update[src_nodes] = timestamps.detach()
            self.last_update[dst_nodes] = timestamps.detach()

        # Classify edge using updated memory
        edge_repr = torch.cat([
            self.memory[src_nodes],
            self.memory[dst_nodes],
            edge_features
        ], dim=1)                          # [B, classifier_input]

        return self.edge_classifier(edge_repr)

    def detach_memory(self):
        """Detach memory from computation graph"""
        self.memory.detach_()

# ============================================================
# M4.3: TRAINING SETUP
# ============================================================
device = torch.device('cpu')

model_tgn = TGN(
    n_nodes      = n_nodes_total,
    edge_feat_dim = train_edge_arr.shape[1],
    memory_dim   = 64,
    hidden_dim   = 64,
    dropout      = 0.3
).to(device)

# Class weight
n_pos         = train_labels_np.sum()
n_neg         = len(train_labels_np) - n_pos
stable_weight = np.sqrt(n_neg / n_pos)

criterion = nn.CrossEntropyLoss(
    weight=torch.tensor(
        [1.0, stable_weight], dtype=torch.float
    )
)

optimizer = torch.optim.Adam(
    model_tgn.parameters(),
    lr=0.001, weight_decay=1e-5
)

print(f"\nModel architecture:")
print(model_tgn)
print(f"\nTotal parameters: "
      f"{sum(p.numel() for p in model_tgn.parameters()):,}")
print(f"Class weight (stable): {stable_weight:.2f}")

# ============================================================
# M4.4: TRAINING LOOP
# Process events in temporal order with mini-batches
# ============================================================
BATCH_SIZE = 2048
N_EPOCHS   = 50

# Convert to tensors
train_src_t  = torch.tensor(train_src,       dtype=torch.long)
train_dst_t  = torch.tensor(train_dst,       dtype=torch.long)
train_ts_t   = torch.tensor(train_ts,        dtype=torch.float)
train_feat_t = torch.tensor(train_edge_arr,  dtype=torch.float)
train_lbl_t  = torch.tensor(train_labels_np, dtype=torch.long)

test_src_t   = torch.tensor(test_src,        dtype=torch.long)
test_dst_t   = torch.tensor(test_dst,        dtype=torch.long)
test_ts_t    = torch.tensor(test_ts,         dtype=torch.float)
test_feat_t  = torch.tensor(test_edge_arr,   dtype=torch.float)
test_lbl_t   = torch.tensor(test_labels_np,  dtype=torch.long)

print(f"\nTraining TGN...")
print(f"  Batch size: {BATCH_SIZE:,}")
print(f"  Epochs    : {N_EPOCHS}")
print(f"  Events/epoch: {len(train_src):,}")

start_time = time.time()
best_f1    = 0
best_state = None
n_batches  = (len(train_src) + BATCH_SIZE - 1) // BATCH_SIZE

for epoch in range(1, N_EPOCHS + 1):

    model_tgn.train()
    model_tgn.reset_memory()  # Reset memory each epoch
    epoch_loss = 0
    all_preds  = []
    all_labels = []

    # Process in temporal order (critical!)
    for batch_idx in range(n_batches):
        start = batch_idx * BATCH_SIZE
        end   = min(start + BATCH_SIZE, len(train_src))

        # Get batch
        src_b   = train_src_t[start:end]
        dst_b   = train_dst_t[start:end]
        ts_b    = train_ts_t[start:end]
        feat_b  = train_feat_t[start:end]
        lbl_b   = train_lbl_t[start:end]

        optimizer.zero_grad()

        # Forward pass
        out  = model_tgn(
            src_b, dst_b, ts_b, feat_b,
            update_memory=True
        )
        loss = criterion(out, lbl_b)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            model_tgn.parameters(), max_norm=1.0
        )
        optimizer.step()
        model_tgn.detach_memory()

        epoch_loss += loss.item()
        all_preds.extend(
            out.argmax(dim=1).detach().cpu().numpy()
        )
        all_labels.extend(lbl_b.cpu().numpy())

    # Epoch metrics
    avg_loss  = epoch_loss / n_batches
    f1_train  = f1_score(
        all_labels, all_preds, zero_division=0
    )

    if epoch % 10 == 0:
        # Evaluate on test set
        model_tgn.eval()
        test_preds  = []
        test_probas = []

        with torch.no_grad():
            n_test_batches = (
                len(test_src) + BATCH_SIZE - 1
            ) // BATCH_SIZE

            for batch_idx in range(n_test_batches):
                start  = batch_idx * BATCH_SIZE
                end    = min(
                    start + BATCH_SIZE, len(test_src)
                )
                src_b  = test_src_t[start:end]
                dst_b  = test_dst_t[start:end]
                ts_b   = test_ts_t[start:end]
                feat_b = test_feat_t[start:end]

                out_b  = model_tgn(
                    src_b, dst_b, ts_b, feat_b,
                    update_memory=False  # Don't update during eval
                )
                test_preds.extend(
                    out_b.argmax(dim=1).cpu().numpy()
                )
                test_probas.extend(
                    torch.softmax(out_b, dim=1)[
                        :, 1
                    ].cpu().numpy()
                )

        f1_test = f1_score(
            test_labels_np, test_preds, zero_division=0
        )

        if f1_test > best_f1:
            best_f1    = f1_test
            best_state = {
                k: v.clone()
                for k, v in model_tgn.state_dict().items()
            }

        print(f"  Epoch {epoch:3d}/{N_EPOCHS} "
              f"| Loss: {avg_loss:.4f} "
              f"| Train F1: {f1_train:.4f} "
              f"| Test F1: {f1_test:.4f}")

train_time_tgn = time.time() - start_time
print(f"\nTraining time: {train_time_tgn:.2f}s")
print(f"Best Test F1 : {best_f1:.4f}")

# ============================================================
# M4.5: FINAL EVALUATION
# ============================================================
print("\nFinal evaluation...")

if best_state is not None:
    model_tgn.load_state_dict(best_state)

model_tgn.eval()
final_preds  = []
final_probas = []

with torch.no_grad():
    n_test_batches = (
        len(test_src) + BATCH_SIZE - 1
    ) // BATCH_SIZE

    for batch_idx in range(n_test_batches):
        start  = batch_idx * BATCH_SIZE
        end    = min(start + BATCH_SIZE, len(test_src))
        src_b  = test_src_t[start:end]
        dst_b  = test_dst_t[start:end]
        ts_b   = test_ts_t[start:end]
        feat_b = test_feat_t[start:end]

        out_b  = model_tgn(
            src_b, dst_b, ts_b, feat_b,
            update_memory=False
        )
        final_preds.extend(
            out_b.argmax(dim=1).cpu().numpy()
        )
        final_probas.extend(
            torch.softmax(out_b, dim=1)[:, 1].cpu().numpy()
        )

y_pred_tgn       = np.array(final_preds)
y_pred_proba_tgn = np.array(final_probas)
y_true_tgn       = test_labels_np

f1_tgn        = f1_score(y_true_tgn, y_pred_tgn,
                          zero_division=0)
precision_tgn = precision_score(y_true_tgn, y_pred_tgn,
                                 zero_division=0)
recall_tgn    = recall_score(y_true_tgn, y_pred_tgn,
                              zero_division=0)
roc_auc_tgn   = roc_auc_score(y_true_tgn, y_pred_proba_tgn)
pr_auc_tgn    = average_precision_score(
    y_true_tgn, y_pred_proba_tgn
)

print(f"\n=== TGN RESULTS ===")
print(f"  F1 Score   : {f1_tgn:.4f}")
print(f"  Precision  : {precision_tgn:.4f}")
print(f"  Recall     : {recall_tgn:.4f}")
print(f"  ROC-AUC    : {roc_auc_tgn:.4f}")
print(f"  PR-AUC     : {pr_auc_tgn:.4f}")
print(f"  Train time : {train_time_tgn:.2f}s")

cm_tgn = confusion_matrix(y_true_tgn, y_pred_tgn)
tn, fp, fn, tp = cm_tgn.ravel()
print(f"\nConfusion Matrix:")
print(f"                 Predicted Normal  Predicted Wash")
print(f"Actual Normal  : {cm_tgn[0][0]:>15,}  {cm_tgn[0][1]:>14,}")
print(f"Actual Wash    : {cm_tgn[1][0]:>15,}  {cm_tgn[1][1]:>14,}")
print(f"\n  TN: {tn:,} | FP: {fp:,} | FN: {fn:,} | TP: {tp:,}")

tgn_results = {
    'model'     : 'TGN',
    'f1'        : f1_tgn,
    'precision' : precision_tgn,
    'recall'    : recall_tgn,
    'roc_auc'   : roc_auc_tgn,
    'pr_auc'    : pr_auc_tgn,
    'train_time': train_time_tgn
}

# ============================================================
# FINAL COMPARISON
# ============================================================
print(f"\n{'='*60}")
print(f"FINAL RESULTS COMPARISON")
print(f"{'='*60}")

all_results = [
    lr_results, rf_results, xgb_results,
    gcn_results, tgn_results
]
comparison_df = pd.DataFrame(
    all_results
).set_index('model')

print(comparison_df[[
    'f1', 'precision', 'recall', 'roc_auc', 'pr_auc'
]].round(4).to_string())

print(f"\n{'='*60}")
print(f"Best model by F1: "
      f"{comparison_df['f1'].idxmax()} "
      f"(F1={comparison_df['f1'].max():.4f})")
print(f"{'='*60}")

print("\nAll models done ✅")
print("Next: M5 — Visualization & Analysis")

M4: TGN (Temporal Graph Network)

Preparing temporal data...
Train: 690,846 events
Test : 296,077 events
Nodes: 152,016
Edge feature dim: 5

Model architecture:
TGN(
  (msg_fn): Sequential(
    (0): Linear(in_features=134, out_features=64, bias=True)
    (1): ReLU()
  )
  (memory_updater): GRUCell(64, 64)
  (edge_classifier): Sequential(
    (0): Linear(in_features=133, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
    (5): Linear(in_features=32, out_features=2, bias=True)
  )
)

Total parameters: 9,773,346
Class weight (stable): 10.50

Training TGN...
  Batch size: 2,048
  Epochs    : 50
  Events/epoch: 690,846
  Epoch  10/50 | Loss: 0.0761 | Train F1: 0.5797 | Test F1: 0.3489
  Epoch  20/50 | Loss: 0.0565 | Train F1: 0.7343 | Test F1: 0.5670
  Epoch  30/50 | Loss: 0.0521 | Train F1: 0.7329 | Test F1: 0.5798
  Epoch  40/50 | Loss: 0.0490 | Train F1: 0.7687 | Test F1: 0.6511

In [19]:
# ============================================================
# DEBUG: Why GCN and TGN underperform?
# ============================================================

print("=== DEBUGGING GCN & TGN PERFORMANCE ===\n")

# ============================================================
# CHECK 1: Label Distribution
# ============================================================
print("CHECK 1: Label Distribution")
print("-" * 40)

print(f"Train wash ratio: {train_labels_np.mean():.4%}")
print(f"Test wash ratio : {test_labels_np.mean():.4%}")

train_wt = train_df[train_df['is_wash_trading'] == 1]
test_wt  = test_df[test_df['is_wash_trading'] == 1]
train_ok = train_df[train_df['is_wash_trading'] == 0]
test_ok  = test_df[test_df['is_wash_trading'] == 0]

print(f"\nWash trading feature comparison (train vs test):")
for col in FEATURE_COLS:
    t_mean = train_wt[col].mean()
    e_mean = test_wt[col].mean()
    diff   = abs(t_mean - e_mean) / (t_mean + 1e-8)
    print(f"  {col:<25}: train={t_mean:.3f}, "
          f"test={e_mean:.3f}, diff={diff:.1%}")

# ============================================================
# CHECK 2: Feature Distribution Shift
# ============================================================
print("\nCHECK 2: Feature Distribution Shift")
print("-" * 40)
print(f"{'Feature':<25} {'Train Mean':>12} "
      f"{'Test Mean':>12} {'Shift':>8}")
print("-" * 60)

for col in FEATURE_COLS:
    t_mean = train_df[col].mean()
    e_mean = test_df[col].mean()
    shift  = abs(t_mean - e_mean) / (t_mean + 1e-8)
    flag   = "⚠️" if shift > 0.3 else "✅"
    print(f"{col:<25} {t_mean:>12.3f} "
          f"{e_mean:>12.3f} {shift:>7.1%} {flag}")

# ============================================================
# CHECK 3: RF Train vs Test
# ============================================================
print("\nCHECK 3: RF Train vs Test Performance")
print("-" * 40)

y_train_pred_rf = rf_model.predict(X_train)
f1_rf_train     = f1_score(y_train, y_train_pred_rf)
f1_rf_test      = f1_score(y_test, y_pred_rf)

print(f"RF Train F1: {f1_rf_train:.4f}")
print(f"RF Test F1 : {f1_rf_test:.4f}")
print(f"Gap        : {f1_rf_train - f1_rf_test:.4f}")

if f1_rf_train - f1_rf_test > 0.1:
    print("⚠️ RF might be overfitting too!")
else:
    print("✅ RF generalizes well")

# ============================================================
# CHECK 4: Manual Label Validation
# ============================================================
print("\nCHECK 4: Manual Label Validation")
print("-" * 40)

sample_wt = test_df[test_df['is_wash_trading'] == 1].head(5)
print("Sample wash trading transactions:")
print(sample_wt[[
    'timestamp_dt', 'from_address', 'to_address',
    'transaction_value', 'num_transitions',
    'pair_frequency', 'is_wash_trading'
]].to_string())

print("\nChecking if these pairs exist in training:")
for _, row in sample_wt.iterrows():
    pair_in_train = train_df[
        (train_df['from_address'] == row['from_address']) &
        (train_df['to_address']   == row['to_address'])
    ]
    from_short = row['from_address'][:8]
    to_short   = row['to_address'][:8]
    print(f"  {from_short}...→{to_short}... "
          f"in train: {len(pair_in_train)} times")

# ============================================================
# CHECK 5: TGN Memory Analysis
# ============================================================
print("\nCHECK 5: TGN Memory Analysis")
print("-" * 40)

model_tgn.eval()
memory_norms = torch.norm(
    model_tgn.memory, dim=1
).detach().cpu().numpy()

print(f"Memory stats after training:")
print(f"  Mean norm : {memory_norms.mean():.4f}")
print(f"  Max norm  : {memory_norms.max():.4f}")
print(f"  Zero nodes: {(memory_norms == 0).sum():,} "
      f"({(memory_norms == 0).mean():.1%})")
print(f"  Non-zero  : {(memory_norms > 0).sum():,}")

wt_wallet_indices = [
    wallet_to_idx[w]
    for w in pd.concat([
        test_wt['from_address'],
        test_wt['to_address']
    ]).unique()
    if w in wallet_to_idx
]

normal_wallet_indices = [
    wallet_to_idx[w]
    for w in pd.concat([
        test_ok['from_address'].head(1000),
        test_ok['to_address'].head(1000)
    ]).unique()
    if w in wallet_to_idx
][:len(wt_wallet_indices)]

wt_mem_norm = torch.norm(
    model_tgn.memory[wt_wallet_indices], dim=1
).mean().item()

normal_mem_norm = torch.norm(
    model_tgn.memory[normal_wallet_indices], dim=1
).mean().item()

print(f"\nMemory norm comparison:")
print(f"  Wash trading wallets : {wt_mem_norm:.4f}")
print(f"  Normal wallets       : {normal_mem_norm:.4f}")
print(f"  Ratio                : {wt_mem_norm/normal_mem_norm:.2f}x")

=== DEBUGGING GCN & TGN PERFORMANCE ===

CHECK 1: Label Distribution
----------------------------------------
Train wash ratio: 0.8983%
Test wash ratio : 0.5475%

Wash trading feature comparison (train vs test):
  transaction_value        : train=437388869133874176.000, test=7980513378978302976.000, diff=1724.6%
  holding_time_hours       : train=144.208, test=335.989, diff=133.0%
  pair_frequency           : train=1.004, test=1.008, diff=0.4%
  time_since_last_hours    : train=55.291, test=59.617, diff=7.8%
  symmetry_ratio_from      : train=0.778, test=1.093, diff=40.5%
  transfers_out_from       : train=2203.397, test=1540.025, diff=30.1%
  transfers_in_from        : train=2148.295, test=1531.053, diff=28.7%
  transfers_out_to         : train=2261.665, test=1650.575, diff=27.0%
  transfers_in_to          : train=2300.064, test=1881.069, diff=18.2%
  num_transitions          : train=144.465, test=332.899, diff=130.4%

CHECK 2: Feature Distribution Shift
------------------------------

In [21]:
# ============================================================
# RESTART: Change Split Strategy
# New approach: August only, split Aug 1-20 (train) 
# and Aug 21-31 (test)
# Rationale: Reduce distribution shift between
# train and test periods
# ============================================================

print("=== RESTARTING WITH NEW SPLIT STRATEGY ===")
print("Train: August 1-20, 2021")
print("Test : August 21-31, 2021")

# ============================================================
# RELOAD: August only from features dataset
# ============================================================
print("\nLoading August data...")

df_aug = pd.read_csv(
    '../dataset/nft_transfers_jun_aug_ready.csv'
)
df_aug['timestamp_dt'] = pd.to_datetime(df_aug['timestamp_dt'])

# Filter August only
df_aug = df_aug[
    df_aug['timestamp_dt'].dt.month == 8
].copy().reset_index(drop=True)

print(f"August total: {len(df_aug):,}")
print(f"Wash trading: {df_aug['is_wash_trading'].sum():,} "
      f"({df_aug['is_wash_trading'].mean():.3%})")

# ============================================================
# FILTER: Top 100 collections (same as before)
# ============================================================
top_100 = df_aug.groupby(
    'nft_address'
).size().sort_values(ascending=False).head(100).index

df_subset = df_aug[
    df_aug['nft_address'].isin(top_100)
].copy().reset_index(drop=True)

print(f"\nAfter top 100 filter: {len(df_subset):,}")
print(f"Wash trading: {df_subset['is_wash_trading'].sum():,} "
      f"({df_subset['is_wash_trading'].mean():.3%})")

# ============================================================
# SPLIT: Temporal 70/30
# ============================================================
df_subset = df_subset.sort_values(
    'timestamp'
).reset_index(drop=True)

split_idx  = int(len(df_subset) * 0.70)
split_time = df_subset.loc[split_idx, 'timestamp_dt']

train_df = df_subset.iloc[:split_idx].copy()
test_df  = df_subset.iloc[split_idx:].copy()

print(f"\n=== NEW TEMPORAL SPLIT ===")
print(f"Split point: {split_time}")
print(f"\nTrain: {len(train_df):,} rows")
print(f"  From: {train_df['timestamp_dt'].min()}")
print(f"  To  : {train_df['timestamp_dt'].max()}")
print(f"  Wash: {train_df['is_wash_trading'].sum():,} "
      f"({train_df['is_wash_trading'].mean():.3%})")
print(f"\nTest : {len(test_df):,} rows")
print(f"  From: {test_df['timestamp_dt'].min()}")
print(f"  To  : {test_df['timestamp_dt'].max()}")
print(f"  Wash: {test_df['is_wash_trading'].sum():,} "
      f"({test_df['is_wash_trading'].mean():.3%})")

# ============================================================
# CHECK: Distribution shift with new split
# ============================================================
print(f"\n=== DISTRIBUTION SHIFT CHECK ===")
print(f"{'Feature':<25} {'Train Mean':>12} "
      f"{'Test Mean':>12} {'Shift':>8}")
print("-" * 62)

FEATURE_COLS = [
    'transaction_value',
    'holding_time_hours',
    'pair_frequency',
    'time_since_last_hours',
    'symmetry_ratio_from',
    'transfers_out_from',
    'transfers_in_from',
    'transfers_out_to',
    'transfers_in_to',
    'num_transitions',
]

for col in FEATURE_COLS:
    t_mean = train_df[col].mean()
    e_mean = test_df[col].mean()
    shift  = abs(t_mean - e_mean) / (t_mean + 1e-8)
    flag   = "⚠️" if shift > 0.3 else "✅"
    print(f"{col:<25} {t_mean:>12.3f} "
          f"{e_mean:>12.3f} {shift:>7.1%} {flag}")

# ============================================================
# PREPARE X, y for tabular models
# ============================================================
LABEL_COL = 'is_wash_trading'

X_train = train_df[FEATURE_COLS].values
y_train = train_df[LABEL_COL].values
X_test  = test_df[FEATURE_COLS].values
y_test  = test_df[LABEL_COL].values

print(f"\nX_train: {X_train.shape}")
print(f"X_test : {X_test.shape}")

# Class weights
from sklearn.utils.class_weight import compute_class_weight
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=y_train
)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}
print(f"\nClass weights:")
print(f"  Normal (0)       : {class_weights[0]:.4f}")
print(f"  Wash trading (1) : {class_weights[1]:.4f}")

=== RESTARTING WITH NEW SPLIT STRATEGY ===
Train: August 1-20, 2021
Test : August 21-31, 2021

Loading August data...


C:\Users\aryak\AppData\Local\Temp\ipykernel_23064\1568829969.py:18: DtypeWarning: Columns (0: token_id) have mixed types. Specify dtype option on import or set low_memory=False.
  df_aug = pd.read_csv(


August total: 1,117,992
Wash trading: 4,000 (0.358%)

After top 100 filter: 721,138
Wash trading: 2,504 (0.347%)

=== NEW TEMPORAL SPLIT ===
Split point: 2021-08-25 05:55:14

Train: 504,796 rows
  From: 2021-08-01 00:00:17
  To  : 2021-08-25 05:55:14
  Wash: 2,214 (0.439%)

Test : 216,342 rows
  From: 2021-08-25 05:55:14
  To  : 2021-08-31 23:59:55
  Wash: 290 (0.134%)

=== DISTRIBUTION SHIFT CHECK ===
Feature                     Train Mean    Test Mean    Shift
--------------------------------------------------------------
transaction_value         788837121902671488.000 1414822014566718208.000   79.4% ⚠️
holding_time_hours             217.320      283.455   30.4% ⚠️
pair_frequency                   1.000        1.002    0.1% ✅
time_since_last_hours          120.885      143.854   19.0% ✅
symmetry_ratio_from              0.709        0.848   19.7% ✅
transfers_out_from            6729.920     7683.497   14.2% ✅
transfers_in_from              250.004      218.866   12.5% ✅
transfers_out

In [22]:
# Quick check transaction_value distribution
print("Transaction value percentiles:")
print(f"  Train p50: {train_df['transaction_value'].quantile(0.50):.2e}")
print(f"  Train p95: {train_df['transaction_value'].quantile(0.95):.2e}")
print(f"  Train p99: {train_df['transaction_value'].quantile(0.99):.2e}")
print(f"  Train max: {train_df['transaction_value'].max():.2e}")
print(f"  Test  p50: {test_df['transaction_value'].quantile(0.50):.2e}")
print(f"  Test  p95: {test_df['transaction_value'].quantile(0.95):.2e}")
print(f"  Test  p99: {test_df['transaction_value'].quantile(0.99):.2e}")
print(f"  Test  max: {test_df['transaction_value'].max():.2e}")

# Check wash trading count
print(f"\nWash trading in test: {y_test.sum():,}")
print(f"Enough for evaluation: {'✅' if y_test.sum() >= 200 else '⚠️'}")

Transaction value percentiles:
  Train p50: 1.90e+17
  Train p95: 2.90e+18
  Train p99: 1.05e+19
  Train max: 1.00e+21
  Test  p50: 3.08e+17
  Test  p95: 5.99e+18
  Test  p99: 1.45e+19
  Test  max: 1.80e+21

Wash trading in test: 290
Enough for evaluation: ✅


In [23]:
# ============================================================
# M4: TGN FIXED
# Key fixes:
# 1. update_memory=True during inference
# 2. No memory reset between train and test
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score,
    confusion_matrix
)
from sklearn.preprocessing import StandardScaler
import time
import numpy as np

print("=" * 55)
print("M4: TGN FIXED")
print("=" * 55)

# ============================================================
# PREPARE DATA
# ============================================================
print("\nPreparing data...")

# Wallet index — from ALL data (train + test)
all_wallets   = pd.concat([
    df_subset['from_address'],
    df_subset['to_address']
]).unique()
wallet_to_idx = {w: i for i, w in enumerate(all_wallets)}
n_nodes       = len(wallet_to_idx)

print(f"Total wallets: {n_nodes:,}")

# Normalize timestamps
ts_min = df_subset['timestamp'].min()
ts_max = df_subset['timestamp'].max()

def normalize_ts(ts):
    return (ts - ts_min) / (ts_max - ts_min + 1e-8)

train_ts = normalize_ts(train_df['timestamp'].values)
test_ts  = normalize_ts(test_df['timestamp'].values)

# Node indices
train_src = train_df['from_address'].map(
    wallet_to_idx
).values
train_dst = train_df['to_address'].map(
    wallet_to_idx
).values
test_src  = test_df['from_address'].map(
    wallet_to_idx
).values
test_dst  = test_df['to_address'].map(
    wallet_to_idx
).values

# Edge features — normalize from train only
FEATURE_COLS = [
    'transaction_value',
    'holding_time_hours',
    'pair_frequency',
    'time_since_last_hours',
    'symmetry_ratio_from',
    'transfers_out_from',
    'transfers_in_from',
    'transfers_out_to',
    'transfers_in_to',
    'num_transitions',
]

edge_scaler    = StandardScaler()
train_edge_arr = edge_scaler.fit_transform(
    train_df[FEATURE_COLS].values.astype(float)
)
test_edge_arr  = edge_scaler.transform(
    test_df[FEATURE_COLS].values.astype(float)
)

train_labels_np = train_df['is_wash_trading'].values
test_labels_np  = test_df['is_wash_trading'].values

print(f"Train: {len(train_src):,} events, "
      f"{train_labels_np.sum():,} wash trading")
print(f"Test : {len(test_src):,} events, "
      f"{test_labels_np.sum():,} wash trading")
print(f"Edge features: {train_edge_arr.shape[1]}")

# ============================================================
# TGN MODEL
# ============================================================
class TGN(nn.Module):
    def __init__(self,
                 n_nodes,
                 edge_feat_dim,
                 memory_dim = 64,
                 hidden_dim = 64,
                 dropout    = 0.3):
        super(TGN, self).__init__()

        self.n_nodes    = n_nodes
        self.memory_dim = memory_dim

        # Memory: one vector per wallet
        self.register_buffer(
            'memory',
            torch.zeros(n_nodes, memory_dim)
        )
        self.register_buffer(
            'last_update',
            torch.zeros(n_nodes)
        )

        # Message function
        msg_dim    = memory_dim * 2 + edge_feat_dim + 1
        self.msg_fn = nn.Sequential(
            nn.Linear(msg_dim, memory_dim),
            nn.ReLU()
        )

        # Memory updater
        self.memory_updater = nn.GRUCell(
            input_size  = memory_dim,
            hidden_size = memory_dim
        )

        # Edge classifier
        clf_input = memory_dim * 2 + edge_feat_dim
        self.edge_classifier = nn.Sequential(
            nn.Linear(clf_input, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 2)
        )

    def reset_memory(self):
        """Reset memory — called only at epoch start"""
        self.memory.zero_()
        self.last_update.zero_()

    def detach_memory(self):
        """Detach memory from computation graph"""
        self.memory = self.memory.detach()
        self.last_update = self.last_update.detach()

    def forward(self,
                src_nodes,
                dst_nodes,
                timestamps,
                edge_features):
        """
        Process temporal edges and ALWAYS update memory.
        This is the key fix — memory is updated both
        during training AND inference.
        """

        # Get current memory
        src_mem = self.memory[src_nodes]
        dst_mem = self.memory[dst_nodes]

        # Time delta since last interaction
        src_time_delta = (
            timestamps - self.last_update[src_nodes]
        ).unsqueeze(1)

        # Compute message
        msg_input = torch.cat([
            src_mem,
            dst_mem,
            edge_features,
            src_time_delta
        ], dim=1)
        msg = self.msg_fn(msg_input)

        # Update memory for source and destination
        new_src_mem = self.memory_updater(msg, src_mem)
        new_dst_mem = self.memory_updater(msg, dst_mem)

        # Write back
        self.memory[src_nodes]     = new_src_mem
        self.memory[dst_nodes]     = new_dst_mem
        self.last_update[src_nodes] = timestamps
        self.last_update[dst_nodes] = timestamps

        # Classify using updated memory
        edge_repr = torch.cat([
            self.memory[src_nodes],
            self.memory[dst_nodes],
            edge_features
        ], dim=1)

        return self.edge_classifier(edge_repr)

# ============================================================
# TRAINING SETUP
# ============================================================
device = torch.device('cpu')

model_tgn = TGN(
    n_nodes       = n_nodes,
    edge_feat_dim = train_edge_arr.shape[1],
    memory_dim    = 64,
    hidden_dim    = 64,
    dropout       = 0.3
).to(device)

# Class weight
n_pos         = train_labels_np.sum()
n_neg         = len(train_labels_np) - n_pos
stable_weight = np.sqrt(n_neg / n_pos)

criterion = nn.CrossEntropyLoss(
    weight=torch.tensor(
        [1.0, stable_weight], dtype=torch.float
    )
)

optimizer = torch.optim.Adam(
    model_tgn.parameters(),
    lr=0.001, weight_decay=1e-5
)

print(f"\nClass weight (stable): {stable_weight:.2f}")
print(f"Parameters: "
      f"{sum(p.numel() for p in model_tgn.parameters()):,}")

# Convert to tensors
train_src_t  = torch.tensor(train_src,       dtype=torch.long)
train_dst_t  = torch.tensor(train_dst,       dtype=torch.long)
train_ts_t   = torch.tensor(train_ts,        dtype=torch.float)
train_feat_t = torch.tensor(train_edge_arr,  dtype=torch.float)
train_lbl_t  = torch.tensor(train_labels_np, dtype=torch.long)

test_src_t   = torch.tensor(test_src,        dtype=torch.long)
test_dst_t   = torch.tensor(test_dst,        dtype=torch.long)
test_ts_t    = torch.tensor(test_ts,         dtype=torch.float)
test_feat_t  = torch.tensor(test_edge_arr,   dtype=torch.float)

# ============================================================
# TRAINING LOOP
# ============================================================
BATCH_SIZE = 2048
N_EPOCHS   = 50
n_batches  = (
    len(train_src) + BATCH_SIZE - 1
) // BATCH_SIZE

print(f"\nTraining TGN (Fixed)...")
print(f"  Epochs    : {N_EPOCHS}")
print(f"  Batch size: {BATCH_SIZE:,}")

start_time = time.time()
best_f1    = 0
best_state = None

for epoch in range(1, N_EPOCHS + 1):

    model_tgn.train()
    model_tgn.reset_memory()  # Reset only at epoch start
    epoch_loss  = 0
    all_preds   = []
    all_labels  = []

    # Process in temporal order
    for b in range(n_batches):
        s = b * BATCH_SIZE
        e = min(s + BATCH_SIZE, len(train_src))

        src_b  = train_src_t[s:e]
        dst_b  = train_dst_t[s:e]
        ts_b   = train_ts_t[s:e]
        feat_b = train_feat_t[s:e]
        lbl_b  = train_lbl_t[s:e]

        optimizer.zero_grad()

        out  = model_tgn(src_b, dst_b, ts_b, feat_b)
        loss = criterion(out, lbl_b)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            model_tgn.parameters(), max_norm=1.0
        )
        optimizer.step()
        model_tgn.detach_memory()

        epoch_loss += loss.item()
        all_preds.extend(
            out.argmax(dim=1).detach().cpu().numpy()
        )
        all_labels.extend(lbl_b.cpu().numpy())

    avg_loss = epoch_loss / n_batches
    f1_train = f1_score(
        all_labels, all_preds, zero_division=0
    )

    if epoch % 10 == 0:

        # ================================================
        # KEY FIX: Evaluate test WITH memory update
        # Continue from training memory state
        # Do NOT reset memory before test evaluation
        # ================================================
        model_tgn.eval()
        test_preds  = []
        test_probas = []

        n_test_b = (
            len(test_src) + BATCH_SIZE - 1
        ) // BATCH_SIZE

        with torch.no_grad():
            for b in range(n_test_b):
                s = b * BATCH_SIZE
                e = min(s + BATCH_SIZE, len(test_src))

                src_b  = test_src_t[s:e]
                dst_b  = test_dst_t[s:e]
                ts_b   = test_ts_t[s:e]
                feat_b = test_feat_t[s:e]

                # update_memory implicitly True
                # because forward() always updates
                out_b  = model_tgn(
                    src_b, dst_b, ts_b, feat_b
                )
                test_preds.extend(
                    out_b.argmax(dim=1).cpu().numpy()
                )
                test_probas.extend(
                    torch.softmax(out_b, dim=1)[
                        :, 1
                    ].cpu().numpy()
                )

        f1_test = f1_score(
            test_labels_np, test_preds, zero_division=0
        )

        if f1_test > best_f1:
            best_f1    = f1_test
            best_state = {
                k: v.clone()
                for k, v in model_tgn.state_dict().items()
            }

        print(f"  Epoch {epoch:3d}/{N_EPOCHS} "
              f"| Loss: {avg_loss:.4f} "
              f"| Train F1: {f1_train:.4f} "
              f"| Test F1: {f1_test:.4f}")

train_time_tgn = time.time() - start_time
print(f"\nTraining time: {train_time_tgn:.2f}s")
print(f"Best Test F1 : {best_f1:.4f}")

# ============================================================
# FINAL EVALUATION
# Load best model, re-run test sequentially
# ============================================================
print("\nFinal evaluation with best model...")

if best_state is not None:
    model_tgn.load_state_dict(best_state)

# Reset and re-run train to restore memory state
model_tgn.eval()
model_tgn.reset_memory()

print("  Restoring memory state from training...")
with torch.no_grad():
    for b in range(n_batches):
        s = b * BATCH_SIZE
        e = min(s + BATCH_SIZE, len(train_src))
        model_tgn(
            train_src_t[s:e],
            train_dst_t[s:e],
            train_ts_t[s:e],
            train_feat_t[s:e]
        )

# Now evaluate test
print("  Evaluating test set...")
final_preds  = []
final_probas = []

with torch.no_grad():
    n_test_b = (
        len(test_src) + BATCH_SIZE - 1
    ) // BATCH_SIZE
    for b in range(n_test_b):
        s = b * BATCH_SIZE
        e = min(s + BATCH_SIZE, len(test_src))

        out_b = model_tgn(
            test_src_t[s:e],
            test_dst_t[s:e],
            test_ts_t[s:e],
            test_feat_t[s:e]
        )
        final_preds.extend(
            out_b.argmax(dim=1).cpu().numpy()
        )
        final_probas.extend(
            torch.softmax(out_b, dim=1)[
                :, 1
            ].cpu().numpy()
        )

y_pred_tgn       = np.array(final_preds)
y_pred_proba_tgn = np.array(final_probas)
y_true_tgn       = test_labels_np

f1_tgn        = f1_score(
    y_true_tgn, y_pred_tgn, zero_division=0
)
precision_tgn = precision_score(
    y_true_tgn, y_pred_tgn, zero_division=0
)
recall_tgn    = recall_score(
    y_true_tgn, y_pred_tgn, zero_division=0
)
roc_auc_tgn   = roc_auc_score(
    y_true_tgn, y_pred_proba_tgn
)
pr_auc_tgn    = average_precision_score(
    y_true_tgn, y_pred_proba_tgn
)

print(f"\n=== TGN FIXED RESULTS ===")
print(f"  F1 Score   : {f1_tgn:.4f}")
print(f"  Precision  : {precision_tgn:.4f}")
print(f"  Recall     : {recall_tgn:.4f}")
print(f"  ROC-AUC    : {roc_auc_tgn:.4f}")
print(f"  PR-AUC     : {pr_auc_tgn:.4f}")
print(f"  Train time : {train_time_tgn:.2f}s")

cm = confusion_matrix(y_true_tgn, y_pred_tgn)
tn, fp, fn, tp = cm.ravel()
print(f"\nConfusion Matrix:")
print(f"                 Predicted Normal  Predicted Wash")
print(f"Actual Normal  : {cm[0][0]:>15,}  {cm[0][1]:>14,}")
print(f"Actual Wash    : {cm[1][0]:>15,}  {cm[1][1]:>14,}")
print(f"\n  TN: {tn:,} | FP: {fp:,} | FN: {fn:,} | TP: {tp:,}")

tgn_results = {
    'model'     : 'TGN (Fixed)',
    'f1'        : f1_tgn,
    'precision' : precision_tgn,
    'recall'    : recall_tgn,
    'roc_auc'   : roc_auc_tgn,
    'pr_auc'    : pr_auc_tgn,
    'train_time': train_time_tgn
}

print(f"\n{'='*55}")
print(f"FINAL COMPARISON")
print(f"{'='*55}")
all_results = [
    lr_results, rf_results, xgb_results,
    gcn_results, tgn_results
]
comp_df = pd.DataFrame(
    all_results
).set_index('model')
print(comp_df[[
    'f1', 'precision', 'recall', 'roc_auc', 'pr_auc'
]].round(4).to_string())

print(f"\nBest model: {comp_df['f1'].idxmax()} "
      f"(F1={comp_df['f1'].max():.4f})")

M4: TGN FIXED

Preparing data...
Total wallets: 128,559
Train: 504,796 events, 2,214 wash trading
Test : 216,342 events, 290 wash trading
Edge features: 10

Class weight (stable): 15.07
Parameters: 44,962

Training TGN (Fixed)...
  Epochs    : 50
  Batch size: 2,048
  Epoch  10/50 | Loss: 0.0481 | Train F1: 0.8769 | Test F1: 0.5662
  Epoch  20/50 | Loss: 0.0451 | Train F1: 0.8808 | Test F1: 0.5923
  Epoch  30/50 | Loss: 0.0428 | Train F1: 0.8735 | Test F1: 0.5738
  Epoch  40/50 | Loss: 0.0415 | Train F1: 0.8459 | Test F1: 0.5945
  Epoch  50/50 | Loss: 0.0387 | Train F1: 0.8688 | Test F1: 0.5540

Training time: 812.50s
Best Test F1 : 0.5945

Final evaluation with best model...
  Restoring memory state from training...
  Evaluating test set...

=== TGN FIXED RESULTS ===
  F1 Score   : 0.5669
  Precision  : 0.6730
  Recall     : 0.4897
  ROC-AUC    : 0.8984
  PR-AUC     : 0.4737
  Train time : 812.50s

Confusion Matrix:
                 Predicted Normal  Predicted Wash
Actual Normal  :   

In [25]:
# ============================================================
# QUICK ANALYSIS: Kenapa TGN masih di bawah RF?
# ============================================================

print("=== TGN vs RF Analysis ===\n")

# 1. Cek wash trading di test — apakah 290 cukup?
print(f"Test wash trading: {test_labels_np.sum()}")
print(f"TP (TGN caught)  : 142 ({142/290:.1%})")
print(f"FN (TGN missed)  : 148 ({148/290:.1%})")
print(f"FP (false alarm) : 69")

# 2. Cek karakteristik wash trading yang di-miss TGN
# vs yang berhasil dideteksi
test_df_copy = test_df.copy()
test_df_copy['tgn_pred'] = y_pred_tgn
test_df_copy['rf_pred']  = y_pred_rf[
    len(y_pred_rf) - len(y_pred_tgn):
] if len(y_pred_rf) > len(y_pred_tgn) else y_pred_rf

wt_caught_tgn = test_df_copy[
    (test_df_copy['is_wash_trading']==1) &
    (test_df_copy['tgn_pred']==1)
]
wt_missed_tgn = test_df_copy[
    (test_df_copy['is_wash_trading']==1) &
    (test_df_copy['tgn_pred']==0)
]

print(f"\nWash trading CAUGHT by TGN (mean features):")
print(wt_caught_tgn[FEATURE_COLS].mean().round(3))

print(f"\nWash trading MISSED by TGN (mean features):")
print(wt_missed_tgn[FEATURE_COLS].mean().round(3))

=== TGN vs RF Analysis ===

Test wash trading: 290
TP (TGN caught)  : 142 (49.0%)
FN (TGN missed)  : 148 (51.0%)
FP (false alarm) : 69

Wash trading CAUGHT by TGN (mean features):
transaction_value        1.724767e+18
holding_time_hours       4.729100e+02
pair_frequency           1.000000e+00
time_since_last_hours    4.523600e+01
symmetry_ratio_from      1.414000e+00
transfers_out_from       1.427817e+03
transfers_in_from        1.427458e+03
transfers_out_to         1.511373e+03
transfers_in_to          1.592408e+03
num_transitions          6.055600e+01
dtype: float64

Wash trading MISSED by TGN (mean features):
transaction_value        2.608390e+18
holding_time_hours       6.015690e+02
pair_frequency           1.007000e+00
time_since_last_hours    9.695400e+01
symmetry_ratio_from      1.398000e+00
transfers_out_from       2.076220e+02
transfers_in_from        1.031280e+02
transfers_out_to         2.470880e+02
transfers_in_to          1.733240e+02
num_transitions          1.824000e+00
